# Visualize payment time, space, and lifecycle behavior

**Goal.** Explore a bounded, deterministic payment world through computed tables and plots.

**Audience.** Data scientists and ML engineers learning FraudTwin's first milestones.

**Prerequisites.** Python 3.12+, FraudTwin, and Polars. Optional plotting cells can install notebook packages.

**Source size.** 1,000 logical payments; outputs are reproducible and temporary.

**Offline path.** Analysis runs without Kafka, databases, or cloud services.


## Optional notebook packages
Run this only for richer plots or t-SNE output.


```bash
!pip install matplotlib scikit-learn
```


## 1. Generate the source world


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import polars as pl

import fraudtwin
from fraudtwin.config import SimulationRunConfig, load_config

root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs" / "minimal.yaml").is_file()
)
base = load_config(root / "configs" / "minimal.yaml")
values = base.model_dump(mode="python")
values["payments"]["daily_target"] = 100
values["simulation"]["duration_days"] = 10
values["population"].update(
    {
        "customers": 100,
        "accounts": 200,
        "cards": 200,
        "merchants": 30,
        "devices": 200,
        "pix_keys": 100,
    }
)
values["behavior"]["amount_max"] = 100
values["simulation"]["seed"] = 2501
config = SimulationRunConfig.model_validate(values)
data = fraudtwin.generate(config)
print({"run_id": data.run_id, "payments": len(data.behavior.payments)})
assert len(data.behavior.payments) >= 1_000

## 2. Analysis step


In [ ]:
payments = pl.DataFrame(
    [
        {
            "payment_id": p.payment_id,
            "rail": p.payment_rail,
            "amount": p.amount,
            "status": p.current_status,
        }
        for p in data.behavior.payments
    ]
)
print(payments.head())
assert payments.height == 1_000

## 3. Analysis step


In [ ]:
rail_counts = payments.group_by("rail").len().sort("len", descending=True)
rail_counts

## 4. Analysis step


In [ ]:
amount_summary = payments.select(
    [
        pl.col("amount").min().alias("min"),
        pl.col("amount").median().alias("median"),
        pl.col("amount").max().alias("max"),
    ]
)
amount_summary

## 5. Analysis step


In [ ]:
event_frame = pl.DataFrame(
    [
        {"event_type": e.event_type, "event_time": e.event_time, "payment_id": e.payment_id}
        for e in data.behavior.payment_events
    ]
)
event_frame.group_by("event_type").len().sort("len", descending=True).head(10)

## 6. Analysis step


In [ ]:
hourly = (
    event_frame.with_columns(pl.col("event_time").dt.hour().alias("hour"))
    .group_by("hour")
    .len()
    .sort("hour")
)
print(hourly)

## 7. Analysis step


In [ ]:
lifecycle = event_frame.group_by(["payment_id", "event_type"]).len()
print({"payments_with_events": lifecycle.select("payment_id").n_unique()})

## 8. Analysis step


In [ ]:
def svg_bars(frame: pl.DataFrame, value: str) -> str:
    rows = frame.to_dicts()
    width, height = 640, 240
    max_value = max((float(row[value]) for row in rows), default=1.0)
    bars = []
    for index, row in enumerate(rows):
        x = 10 + index * (width - 20) / max(1, len(rows))
        bar_height = 180 * float(row[value]) / max_value
        bars.append(
            f'<rect x="{x:.1f}" y="{200-bar_height:.1f}" width="18" height="{bar_height:.1f}"/>'
        )
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="640" height="240">'
        + "".join(bars)
        + "</svg>"
    )


print(svg_bars(hourly, "len")[:80])

## 9. Analysis step


In [ ]:
with TemporaryDirectory(prefix="fraudtwin-viz-25-") as tmp:
    plot_path = Path(tmp) / "hourly-events.svg"
    plot_path.write_text(svg_bars(hourly, "len"), encoding="utf-8")
    print({"plot": str(plot_path), "bytes": plot_path.stat().st_size})
    assert plot_path.stat().st_size > 100

## 10. Analysis step


In [ ]:
fingerprint = fraudtwin.aggregate_fingerprint([dict(row) for row in payments.to_dicts()])
print({"payment_fingerprint": fingerprint})
assert isinstance(fingerprint, str) and fingerprint